# Ch.1 — Recommender Systems Fundamentals

> The simplest recommender: rank items by popularity and recommend the same list to everyone. This establishes the evaluation framework and baseline for the entire track.

**Dataset:** MovieLens 100k — 100,000 ratings from 943 users on 1,682 movies  
**Task:** Build a popularity baseline and measure hit rate@10  
**Outcome:** Popularity baseline = ~42% HR@10

## The Core Idea

A recommender system predicts which items a user will prefer. Three fundamental approaches:

1. **Content-Based**: Recommend items similar to what you liked (using item features)
2. **Collaborative Filtering**: Recommend items that similar users liked
3. **Hybrid**: Combine both

This chapter builds the simplest approach — **popularity baseline** — and establishes evaluation metrics (Precision@k, Recall@k, HR@k, NDCG@k) used throughout the track.

## Running Example

You are a data scientist at **FlixAI**, a movie streaming platform. The VP of Product says: "Give me a recommendation widget. I don't care how it works — just show users movies they'll actually watch."

**Grand Challenge — 5 Constraints:**
1. ACCURACY: >85% hit rate@10
2. COLD START: Handle new users/items
3. SCALABILITY: 1M+ ratings
4. DIVERSITY: Not just popular movies
5. EXPLAINABILITY: "Because you liked X"

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

sns.set_theme(style="whitegrid", palette="muted")
SEED = 42
np.random.seed(SEED)

print("Libraries loaded.")

In [ ]:
# ── Load MovieLens 100k ───────────────────────────────────────────────────
url = "https://files.grouplens.org/datasets/movielens/ml-100k/"

ratings = pd.read_csv(
    url + "u.data", sep="\t",
    names=["user_id", "item_id", "rating", "timestamp"]
)

movies = pd.read_csv(
    url + "u.item", sep="|", encoding="latin-1", header=None,
    names=["item_id", "title", "release_date", "video_release", "url",
           "unknown", "Action", "Adventure", "Animation", "Children",
           "Comedy", "Crime", "Documentary", "Drama", "Fantasy",
           "Film-Noir", "Horror", "Musical", "Mystery", "Romance",
           "Sci-Fi", "Thriller", "War", "Western"],
    usecols=range(24)
)

print(f"Ratings: {len(ratings):,}")
print(f"Users:   {ratings['user_id'].nunique()}")
print(f"Movies:  {ratings['item_id'].nunique()}")
print(f"Sparsity: {1 - len(ratings)/(943*1682):.1%}")
ratings.head()

In [ ]:
# ── Exploratory Data Analysis ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Rating distribution
axes[0].hist(ratings["rating"], bins=5, color="#2980b9", edgecolor="white", rwidth=0.8)
axes[0].set(title="Rating Distribution", xlabel="Rating", ylabel="Count")

# Ratings per user
user_counts = ratings.groupby("user_id").size()
axes[1].hist(user_counts, bins=50, color="#27ae60", edgecolor="white")
axes[1].set(title="Ratings per User", xlabel="# Ratings", ylabel="# Users")
axes[1].axvline(user_counts.median(), color="red", linestyle="--", label=f"Median={user_counts.median():.0f}")
axes[1].legend()

# Ratings per movie
item_counts = ratings.groupby("item_id").size()
axes[2].hist(item_counts, bins=50, color="#e67e22", edgecolor="white")
axes[2].set(title="Ratings per Movie", xlabel="# Ratings", ylabel="# Movies")
axes[2].axvline(item_counts.median(), color="red", linestyle="--", label=f"Median={item_counts.median():.0f}")
axes[2].legend()

plt.suptitle("MovieLens 100k — Data Overview", y=1.02, fontsize=14)
plt.tight_layout()
plt.savefig("img/data_overview.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Avg ratings/user: {user_counts.mean():.0f}")
print(f"Avg ratings/movie: {item_counts.mean():.0f}")

## Evaluation Framework

We use **leave-one-out** evaluation: for each user, hold out the **last** rating (by timestamp) as the test item. The model must rank this item in the top-k among 99 random negatives + 1 positive.

**Key Metrics:**
- **Hit Rate@k**: Fraction of users where the test item appears in top-k
- **NDCG@k**: Rewards higher-ranked hits more than lower-ranked ones
- **Precision@k**: Fraction of top-k items that are relevant

In [ ]:
# ── Leave-One-Out Train/Test Split ────────────────────────────────────────
def leave_one_out_split(ratings_df):
    """Hold out the last rating per user (by timestamp) for testing."""
    ratings_sorted = ratings_df.sort_values("timestamp")
    test = ratings_sorted.groupby("user_id").tail(1).copy()
    train = ratings_sorted.drop(test.index).copy()
    return train, test

train, test = leave_one_out_split(ratings)
print(f"Train: {len(train):,} ratings")
print(f"Test:  {len(test):,} ratings (1 per user)")
print(f"Users in test: {test['user_id'].nunique()}")

In [ ]:
# ── Evaluation Metrics ────────────────────────────────────────────────────
def hit_rate_at_k(test_df, top_k_per_user, k=10):
    """Fraction of users with ≥1 test item in their top-k recommendations."""
    hits = 0
    for _, row in test_df.iterrows():
        user = row["user_id"]
        test_item = row["item_id"]
        recs = top_k_per_user.get(user, [])[:k]
        if test_item in recs:
            hits += 1
    return hits / len(test_df)

def ndcg_at_k(test_df, top_k_per_user, k=10):
    """Average NDCG@k across all test users."""
    ndcgs = []
    for _, row in test_df.iterrows():
        user = row["user_id"]
        test_item = row["item_id"]
        recs = top_k_per_user.get(user, [])[:k]
        if test_item in recs:
            rank = recs.index(test_item) + 1
            ndcgs.append(1.0 / np.log2(rank + 1))
        else:
            ndcgs.append(0.0)
    return np.mean(ndcgs)

print("Evaluation functions defined.")

## Popularity Baseline

The simplest recommender: rank movies by **Bayesian average** rating and recommend the same top-k to everyone.

$$\text{score}(i) = \frac{|U_i| \cdot \bar{r}_i + C \cdot \mu}{|U_i| + C}$$

- $|U_i|$: number of ratings for item $i$
- $\bar{r}_i$: average rating for item $i$
- $\mu$: global mean rating
- $C$: damping constant (median ratings per item)

In [ ]:
# ── Popularity Baseline ───────────────────────────────────────────────────
item_stats = train.groupby("item_id")["rating"].agg(["mean", "count"])
global_mean = train["rating"].mean()
C = item_stats["count"].median()

item_stats["bayesian_avg"] = (
    (item_stats["count"] * item_stats["mean"] + C * global_mean)
    / (item_stats["count"] + C)
)

# Sort by Bayesian average
top_items_sorted = item_stats.sort_values("bayesian_avg", ascending=False).index.tolist()

# For each user: recommend top-k items they haven't rated
user_rated_train = train.groupby("user_id")["item_id"].apply(set).to_dict()

top_k_pop = {}
for user_id in test["user_id"].unique():
    rated = user_rated_train.get(user_id, set())
    recs = [item for item in top_items_sorted if item not in rated][:10]
    top_k_pop[user_id] = recs

hr = hit_rate_at_k(test, top_k_pop, k=10)
ndcg = ndcg_at_k(test, top_k_pop, k=10)

print(f"Popularity Baseline:")
print(f"  HR@10   = {hr:.3f} ({hr*100:.1f}%)")
print(f"  NDCG@10 = {ndcg:.4f}")

In [ ]:
# ── Top-10 Most Popular Movies ────────────────────────────────────────────
top_10 = item_stats.sort_values("bayesian_avg", ascending=False).head(10)
top_10_with_titles = top_10.merge(
    movies[["item_id", "title"]], left_index=True, right_on="item_id"
)

print("Top-10 Movies by Bayesian Average:")
for i, row in top_10_with_titles.iterrows():
    print(f"  {row['title']:<45} avg={row['mean']:.2f}  count={row['count']:.0f}  bayesian={row['bayesian_avg']:.3f}")

In [ ]:
# ── Visualise: Bayesian Average vs Raw Average ────────────────────────────
fig, ax = plt.subplots(figsize=(10, 6))

scatter = ax.scatter(
    item_stats["count"], item_stats["mean"],
    c=item_stats["bayesian_avg"], cmap="RdYlGn", alpha=0.6, s=15, edgecolors="none"
)
ax.set(xlabel="Number of Ratings", ylabel="Raw Average Rating",
       title="Bayesian Average Shrinks Low-Count Movies Toward Global Mean")
ax.axhline(global_mean, color="gray", linestyle="--", alpha=0.5, label=f"Global mean = {global_mean:.2f}")
ax.legend()
plt.colorbar(scatter, ax=ax, label="Bayesian Average")
plt.tight_layout()
plt.savefig("img/bayesian_average.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── HR@k Curve: How does hit rate change with k? ──────────────────────────
k_values = [1, 3, 5, 10, 20, 50, 100]
hr_values = []

for k in k_values:
    # Extend recommendations for larger k
    top_k_extended = {}
    for user_id in test["user_id"].unique():
        rated = user_rated_train.get(user_id, set())
        recs = [item for item in top_items_sorted if item not in rated][:k]
        top_k_extended[user_id] = recs
    hr_values.append(hit_rate_at_k(test, top_k_extended, k=k))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(k_values, [h * 100 for h in hr_values], "o-", color="#2980b9", linewidth=2, markersize=8)
ax.axhline(85, color="red", linestyle="--", alpha=0.7, label="Target: 85% HR@10")
ax.set(xlabel="k (number of recommendations)", ylabel="Hit Rate@k (%)",
       title="Popularity Baseline: Hit Rate vs k")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("img/hr_at_k_curve.png", dpi=150, bbox_inches="tight")
plt.show()

for k, hr in zip(k_values, hr_values):
    print(f"  HR@{k:>3d} = {hr:.3f} ({hr*100:.1f}%)")

In [ ]:
# ── Genre Distribution of Top-10 Recommendations ─────────────────────────
genre_cols = ["Action", "Adventure", "Animation", "Children", "Comedy",
              "Crime", "Documentary", "Drama", "Fantasy", "Film-Noir",
              "Horror", "Musical", "Mystery", "Romance", "Sci-Fi",
              "Thriller", "War", "Western"]

top_10_ids = item_stats.sort_values("bayesian_avg", ascending=False).head(10).index
top_10_movies = movies[movies["item_id"].isin(top_10_ids)]
genre_counts = top_10_movies[genre_cols].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
genre_counts[genre_counts > 0].plot(kind="barh", ax=ax, color="#27ae60", edgecolor="white")
ax.set(xlabel="Count in Top-10", title="Genre Distribution of Popularity Baseline Top-10")
plt.tight_layout()
plt.savefig("img/genre_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

print("Note: Popularity baseline heavily favours Drama — low diversity!")

## Progress Check

| # | Constraint | Target | Ch.1 Status |
|---|-----------|--------|-------------|
| 1 | ACCURACY | >85% HR@10 | ❌ ~42% |
| 2 | COLD START | New users/items | ⚠️ Same list for everyone |
| 3 | SCALABILITY | 1M+ ratings | ✅ Precomputed list |
| 4 | DIVERSITY | Not just popular | ❌ Only popular movies |
| 5 | EXPLAINABILITY | "Because you liked X" | ❌ "Because it's popular" |

**Bottom line**: 42% hit rate — the simplest baseline. Everyone gets the same 10 movies. We need **personalisation** to go further.

**Next**: Ch.2 — Collaborative Filtering → find users with similar taste and recommend what *they* liked.

## Exercises

**Exercise 1 — Raw Average vs Bayesian Average**  
Build a popularity baseline using raw average rating (no Bayesian damping). Compare HR@10 against the Bayesian baseline. Which is better and why?

**Exercise 2 — Rating Count Baseline**  
Instead of average rating, rank movies by total number of ratings (most-rated = most popular). Compare HR@10. Does "most rated" outperform "highest rated"?

**Exercise 3 — Genre-Specific Popularity**  
Build a popularity baseline that's genre-aware: for users whose most-rated genre is Sci-Fi, recommend the top Sci-Fi movies. Compare HR@10 and diversity against the global popularity baseline.

In [ ]:
# ── Exercise 1 scaffold — Raw Average vs Bayesian ─────────────────────────
# TODO: Build popularity baseline using raw mean (no damping)
# Compare HR@10 with Bayesian baseline

# raw_top_items = item_stats.sort_values("mean", ascending=False).index.tolist()
# ... build top_k_raw, evaluate

pass

In [ ]:
# ── Exercise 2 scaffold — Rating Count Baseline ──────────────────────────
# TODO: Rank by number of ratings instead of average rating
# Compare HR@10

# count_top_items = item_stats.sort_values("count", ascending=False).index.tolist()
# ... build top_k_count, evaluate

pass

In [ ]:
# ── Exercise 3 scaffold — Genre-Specific Popularity ──────────────────────
# TODO: For each user, find their most-rated genre, then recommend
# top movies from that genre only. Compare HR@10 and diversity.

# Steps:
# 1. Merge train ratings with movie genres
# 2. For each user, find dominant genre
# 3. Build genre-specific popularity rankings
# 4. Evaluate HR@10

pass